## **Table of Contents**
1. **Setup**  
   - Configure Spark and GeoMesa  
   - Initialize Spark Session  

2. **Data Ingestion**  
   - Insert Airport Data into HBase  
   - Simulate Drone Movement Data  

3. **Data Processing & Analysis**  
   - Query and Filter Drone Data  
   - Identify Drones Near Airports  

4. **Visualization & Insights**  
   - Visualize All Tracked Drones  
   - Highlight Suspicious Drones (Violating Airspace)

## Setup

In [ ]:
spark_home = '/Users/vhegde/Downloads/HAC-693/spark-3.5.2-bin-hadoop3-scala2.13'
geomesa_hbase_spark_jars_path = '/Users/vhegde/Downloads/HAC-693/geomesa-hbase-spark-runtime-hbase2_2.13-5.0.2-SNAPSHOT.jar'

In [ ]:
import geomesa_pyspark

conf = geomesa_pyspark.configure(
    jars=[geomesa_hbase_spark_jars_path],
    packages=['geomesa_pyspark', 'pytz'],
    spark_home=spark_home
).setAppName('MyTestApp')

conf.set('spark.master', 'local')

from pyspark.sql import SparkSession

# Create the Spark session
spark = (
    SparkSession
    .builder
    .config(conf=conf)
    .enableHiveSupport()
    .getOrCreate()
)

## Insert Airport Data

In [ ]:
# create schema for airport data
import subprocess

create_schema_cmd = [
    "geomesa-hbase", "create-schema",
    "-c", "airport",
    "-f", "airport_f",
    "-s", "airport_id:Integer,airport_name:String,airport_city:String,airport_country:String,iata:String,icao:String,latitude:Double,longitude:Double,elevation:Integer,timezone_offset:Integer,dst:String,timezone:String,type:String,airport_category:String,*geom:Point:srid=4326"
]
result = subprocess.run(create_schema_cmd, capture_output=True, text=True)
print(result)

In [ ]:
# ingest airport data
ingest_cmd = [
    "geomesa-hbase", "ingest",
    "-c", "airport",
    "-f", "airport_f",
    "airports-extended.dat",
    "--force"
]
result = subprocess.run(ingest_cmd, capture_output=True, text=True)
print(result)

In [ ]:
# we need to flush hbase table to parsist the data
flush_cmd = [
    "hbase", "shell", "flush_all_tables.hbase"
]

result = subprocess.run(flush_cmd, capture_output=True, text=True)
print(result)

In [ ]:
airport_params = {
    "hbase.zookeepers": "hbase-docker",
    "hbase.catalog": "airport"
}

airport_df = ( spark
    .read
    .format("geomesa")
    .options(**airport_params)
    .option("geomesa.feature", "airport_f")
    .load()
)
airport_df.createOrReplaceTempView("airports")

In [ ]:
# a placeholder to install the packages
pip install pandas

In [ ]:
# create schema for drone data
create_schema_cmd = [
    "geomesa-hbase", "create-schema",
    "-c", "drone",
    "-f", "drone_movement_f",
    "-s", "drone_id:String,description:String,latitude:Double,longitude:Double,timestamp:Date,PotentialThreat:Boolean,*geom:Point:srid=4326"
]
result = subprocess.run(create_schema_cmd, capture_output=True, text=True)
print(result)

## Drone Movement Data Simulation

In [ ]:
import folium
import random
import time
import math
import pandas as pd
from datetime import datetime, timedelta

boundary_range_km = 50
km_change_per_degree = 111
drone_descriptions = ['Surveillance', 'Attack', 'Inspection', 'Survey']

file_name = "drone_movement_data.csv"

airport_locations = spark.sql("""
    select latitude, longitude
    from airports
    where airport_country='India' and airport_city='Mumbai' and type='airport'
""").rdd.map(lambda row: {"latitude": row["latitude"], "longitude": row["longitude"]}).collect()

min_lat = min([point['latitude'] for point in airport_locations])
max_lat = max([point['latitude'] for point in airport_locations])
min_lon = min([point['longitude'] for point in airport_locations])
max_lon = max([point['longitude'] for point in airport_locations])


buffer_lat = boundary_range_km / km_change_per_degree  
buffer_lon = boundary_range_km / (km_change_per_degree * abs(math.cos(math.radians((min_lat + max_lat) / 2))))

# Boundary
south_bound = min_lat - buffer_lat
north_bound = max_lat + buffer_lat
west_bound = min_lon - buffer_lon
east_bound = max_lon + buffer_lon

command = [
    'python', 'drone_data_generator.py', 
    str(south_bound), 
    str(north_bound), 
    str(west_bound), 
    str(east_bound), 
    str(file_name)
]

with open("drone_data_generator_output.log", "w") as output_file, open("drone_data_generator_error.log", "w") as error_file:
    subprocess.Popen(command, stdout=output_file, stderr=error_file)

In [ ]:
drone_params = {
    "hbase.zookeepers": "hbase-docker",
    "hbase.catalog": "drone"
}

drone_df = ( spark
    .read
    .format("geomesa")
    .options(**drone_params)
    .option("geomesa.feature", "drone_movement_f")
    .load()
)
drone_df.createOrReplaceTempView("drones")

## Visualize All Tracked Drones

In [ ]:
drone_data = spark.sql("""
    SELECT *
    FROM drones
""").toPandas()

colors = ['blue', 'red', 'green', 'orange', 'purple', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'yellow', 'pink', 'brown', 'gray', 'black']
current_color_index = 0
no_fly_zone_in_meters = 4000


# Create a folium map centered on the combined area (average of latitudes and longitudes)
combined_map = folium.Map(location=[(min_lat + max_lat) / 2, (min_lon + max_lon) / 2], zoom_start=10)

# Plot the airport locations
for loc in airport_locations:
    folium.Marker(
        location=[loc['latitude'], loc['longitude']],
        popup=f"Latitude: {loc['latitude']}, Longitude: {loc['longitude']}"
    ).add_to(combined_map)
    
    # Add no-fly zone circle with 4 km radius
    folium.Circle(
        radius=no_fly_zone_in_meters,  # 4 km radius
        location=[loc['latitude'], loc['longitude']],
        color="red",
        fill=True,
        fill_opacity=0.2,
        popup="No-fly zone (4 km radius)"
    ).add_to(combined_map)

# Plot the drone flight paths
grouped = drone_data.groupby('drone_id')

for drone_id, group in grouped:
    color = colors[current_color_index]
    current_color_index += 1
    
    # Get the flight path for the drone
    flight_path = [(row['latitude'], row['longitude']) for index, row in group.iterrows()]
    
    # Plot the flight path with the unique color
    folium.PolyLine(flight_path, color=color, weight=2.5, opacity=1, popup=f"{drone_id}").add_to(combined_map)
    
combined_map

## Visualize Suspicious Drones

In [ ]:
from pyspark.sql import functions as F

filtered_airports_df = airport_df.filter(
    (F.col("airport_country") == "India") &
    (F.col("airport_city") == "Mumbai") &
    (F.col("type") == "airport")
)

buffered_airports_df = filtered_airports_df.withColumn(
    "airport_buffer", F.expr("st_bufferPoint(geom, 4000)")
)

buffered_airports_df.createOrReplaceTempView("buffered_airports")

distinct_drones_query = """
SELECT DISTINCT d.drone_id
FROM drones d
JOIN buffered_airports a
ON ST_Within(d.geom, a.airport_buffer)
"""

distinct_drones_df = spark.sql(distinct_drones_query)
distinct_drones_df.show()

distinct_drones_count_query = """
SELECT COUNT(DISTINCT d.drone_id) AS unique_drones_count
FROM drones d
JOIN buffered_airports a
ON ST_Within(d.geom, a.airport_buffer)
"""

unique_drones_count_df = spark.sql(distinct_drones_count_query)
unique_drones_count = unique_drones_count_df.collect()[0]["unique_drones_count"]

print(f"Number of unique drones near airports: {unique_drones_count}")

In [ ]:
drone_data = spark.sql(f"""
    SELECT *
    FROM drones
    Where drones.drone_id IN ({distinct_drones_query})
""").toPandas()

colors = ['blue', 'red', 'green', 'orange', 'purple', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'yellow', 'pink', 'brown', 'gray', 'black']
current_color_index = 0
no_fly_zone_in_meters = 4000


# Create a folium map centered on the combined area (average of latitudes and longitudes)
combined_map = folium.Map(location=[(min_lat + max_lat) / 2, (min_lon + max_lon) / 2], zoom_start=10)


# Plot the airport locations
for loc in airport_locations:
    folium.Marker(
        location=[loc['latitude'], loc['longitude']],
        popup=f"Latitude: {loc['latitude']}, Longitude: {loc['longitude']}"
    ).add_to(combined_map)
    
    # Add no-fly zone circle with 4 km radius
    folium.Circle(
        radius=no_fly_zone_in_meters,  # 4 km radius
        location=[loc['latitude'], loc['longitude']],
        color="red",
        fill=True,
        fill_opacity=0.2,
        popup="No-fly zone (4 km radius)"
    ).add_to(combined_map)

# Plot the drone flight paths
grouped = drone_data.groupby('drone_id')

for drone_id, group in grouped:
    color = colors[current_color_index]
    current_color_index += 1
    
    # Get the flight path for the drone
    flight_path = [(row['latitude'], row['longitude']) for index, row in group.iterrows()]
    
    # Plot the flight path with the unique color
    folium.PolyLine(flight_path, color=color, weight=2.5, opacity=1, popup=f"{drone_id}").add_to(combined_map)
    
combined_map